# 02 — Qualité des données & hallucinations

**Cadre :** avant tout modèle, auditer la fiabilité des sources cliniques hétérogènes (sujet CISIA).

Ce notebook documente **toutes** les anomalies injectées volontairement dans les données synthétiques :

| Catégorie | Définition | Où c'est détecté |
|-----------|------------|------------------|
| **Trous** | Valeurs manquantes / vides | `quality_report.py` |
| **Aberrantes** | Hors plage physiologique ou biologique | `vitals.py`, `quality.py` |
| **Mal notées** | Incohérences métier (« hallucinations ») | durée séjour, TAD≥TAS, actes hors séjour… |
| **Sensibilité IA** | Colonnes RGPD & usage modèle | `registre.py` |

Synthèse complète (nulls + dates + patho) : [05_analyse_donnees](05_analyse_donnees.ipynb)  
Rapport HTML : `data/curated/qualite/rapport_qualite.html`


In [ ]:
import runpy
from pathlib import Path
_sp = next(a / "setup_path.py" for a in [Path.cwd(), *Path.cwd().parents] if (a / "setup_path.py").is_file())
runpy.run_path(str(_sp))

%matplotlib inline
from _utils.bootstrap import ROOT, paths, COLORS
from _utils.qualite_data import collecter_exemples_hallucinations
from _utils.qualite_plots import plot_dataset_qualite, plot_sensibilite_ia, plot_synthese_globale
from src.data.quality_report import export_quality_report, run_quality_report
from IPython.display import display, Markdown
import pandas as pd

report = run_quality_report(paths)
print(f"Fichiers analysés : {len(report['datasets'])}")


## Synthèse globale (tous fichiers)

In [ ]:
plot_synthese_globale(report)

## Détail par fichier CSV

Pour chaque source : 4 graphiques (trous, aberrants, mal notés, synthèse) + sensibilité IA si applicable.

In [ ]:
for ds in report["datasets"]:
    display(Markdown(f"### `{ds['fichier']}` — {ds['n_lignes']:,} lignes"))
    plot_dataset_qualite(ds)
    if ds.get("sensibilite_ia"):
        plot_sensibilite_ia(ds["sensibilite_ia"], fichier=ds["fichier"])


## Exemples concrets de lignes problématiques

Tableaux issus du code métier (`qualite_data.py`) — ce sont les **hallucinations** à expliquer au jury.


In [ ]:
exemples = collecter_exemples_hallucinations(paths)
for nom, df in exemples.items():
    display(Markdown(f"#### `{nom}` ({len(df)} lignes affichées)"))
    display(df)
if not exemples:
    print("Aucun exemple — vérifier les flags dans quality.py / vitals.py")


## Légende des types d'anomalie

In [ ]:
legende = pd.DataFrame([
    {"type": "sejours_duree_incoherente", "description": "DureeSejour ≠ dates admission/sortie", "gravité": "mal notée"},
    {"type": "signes_vitaux_exclus", "description": "FC/TAS/TAD hors plage ou TAD ≥ TAS", "gravité": "aberrante + exclusion modèle"},
    {"type": "biologies_aberrantes", "description": "Valeur labo hors références ×2", "gravité": "aberrante"},
    {"type": "objets_connectes_signal_faible", "description": "QualiteSignal ≠ Bon", "gravité": "fiabilité télésurveillance"},
    {"type": "actes_date_hors_sejour", "description": "DateActe avant admission ou après sortie", "gravité": "hallucination temporelle"},
])
legende


## Export rapport HTML

In [ ]:
out = export_quality_report(paths)
for k, p in out.items():
    print(f"{k}: {p}")
